<a href="https://colab.research.google.com/github/RuiXuePitt/CGSimFinetune/blob/main/ColabNB/AskCGsim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#INSTALL NECESSARY PACKAGE

In [1]:
pip install -U bitsandbytes>=0.46.1

#LOGIN

In [2]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from pathlib import Path
from google.colab import output

output.enable_custom_widget_manager()
login(userdata.get('HugFace'))
curr_path = Path.cwd()

#IMPORTS

In [3]:
from peft import PeftModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [4]:
import sys
sys.path.append("./drive/MyDrive/resources/CGSim_Train")
import DBTool as dbt

In [5]:
import sqlite3
dbpath = "./drive/MyDrive/resources/CGSim_Train/CGsimSite.db"
conn = sqlite3.connect(dbpath)
cursor = conn.cursor()

In [6]:
import json
train_data = []
with open("drive/MyDrive/resources/CGSim_Train/traindata.jsonl", "r") as f:
  for line in f:
    train_data.append(json.loads(line))

#LOAD Quantized Model with PEFT

In [7]:
base_model_id = "AI4SciNoob/Llama-3.1-Nemotron-Nano-8B-v1-AskCGSim"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
ft_model = PeftModel.from_pretrained(base_model, "./drive/MyDrive/nemotron-llama8b-CGsim/checkpoint-250")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [10]:
print("allocated GiB:", torch.cuda.memory_allocated()/1024**3)
print("reserved  GiB:", torch.cuda.memory_reserved()/1024**3)
print("is_loaded_in_4bit:", getattr(ft_model, "is_loaded_in_4bit", None))
print("is_loaded_in_8bit:", getattr(ft_model, "is_loaded_in_8bit", None))

allocated GiB: 5.361886978149414
reserved  GiB: 14.9609375
is_loaded_in_4bit: True
is_loaded_in_8bit: None


In [12]:
ft_model.config.use_cache = True

#Functions

In [13]:
def gen_new_text(model, messages):
    prompt = tokenizer.apply_chat_template(
      messages,
      tools=train_data[0]["tools"],
      tokenize=False,
      add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=256,
            do_sample=False,
        )

    prompt_len = inputs["input_ids"].shape[1]
    new_ids = out[0][prompt_len:]
    return tokenizer.decode(new_ids, skip_special_tokens=False)

In [14]:
def extract_toolcall_block(text: str):
    s = text.find("<TOOLCALL>[")
    if s == -1:
        return "", None
    e = text.find("]</TOOLCALL>", s)
    if e == -1:
        return "", None

    array_json = text[s + len("<TOOLCALL>") : e + 1]
    toolcall_block = json.loads(array_json)[0] # dict

    return text[:s], toolcall_block

In [15]:
def ask_cgsim(model, question, max_rounds=6):
    print("Question: ", question, "\n")
    messages = [
      {'role': 'system',
      'content': 'You are a CGsim agent. Answer questions related to grid simulation related questions. Use tools when needed.'},
      {'role': 'user',
      'content': question}]

    for step in range(max_rounds):
        gen_text = gen_new_text(model, messages)
        messages.append({'role': 'assistant', 'content': gen_text})

        think, toolcall_block = extract_toolcall_block(gen_text)
        if toolcall_block is None:
            print("Answer \n", gen_text.split("<eot_id>")[0])
            return {'answer': gen_text.split("<eot_id>")[0], 'messages': messages}

        tool_name = toolcall_block["name"]
        tool_args = ""

        if tool_name.startswith("check_"):
            tool_result = getattr(dbt, tool_name)(cursor)
        elif tool_name == ("execute_sql"):
            tool_args = toolcall_block["arguments"]["sql"]
            print(tool_args)
            tool_result = dbt.execute_sql(cursor, tool_args)
        else:
            raise Exception(f"Unknown tool name: {tool_name}")

        print("Thinking \n", think, "\n")
        print("Tool Call \n", tool_name, tool_args, "\n")
        messages.append({'role': 'tool', 'content': json.dumps(tool_result, ensure_ascii=False)})

    return {'answer': None, 'messages': messages}


In [16]:
ans = ask_cgsim(ft_model, "During execution of job 2794720992, what is the total queue time?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question:  During execution of job 2794720992, what is the total queue time? 

Thinking 
 Let's think step by step.
According to 'execution', the EVENT type may be JobExecution but is not certain.
Therefore, all available event types and their data structures should be checked.

 

Tool Call 
 check_All  

SELECT DISTINCT json_extract(METADATA, '$.total_queue_time') AS total_queue_time FROM EVENTS WHERE EVENT = 'JobExecution' AND JOB_ID = 2794720992 AND json_extract(METADATA, '$.total_queue_time') IS NOT NULL;
Thinking 
 Let's think step by step.
According to checked data structure, 'total queue time' may be related to 'total_queue_time' in EVENT = 'JobExecution'.
According to '2794720992', JOB_ID may be used for filtering.
According to 'total queue time', use json_extract(METADATA, '$.total_queue_time') to retrieve the value.

 

Tool Call 
 execute_sql SELECT DISTINCT json_extract(METADATA, '$.total_queue_time') AS total_queue_time FROM EVENTS WHERE EVENT = 'JobExecution' AND JOB_ID 

In [ ]:
ans = ask_cgsim(ft_model, "What is the possible reason that execution of job 8774052003 takes so long time, could you please explain?")

Question:  What is the possible reason that execution of job 8774052003 takes so long time, could you please explain? 

Thinking 
 Let's think step by step.
According to 'execution', the EVENT type may be JobExecution but is not certain.
Therefore, all available event types and their data structures should be checked.

 

Tool Call 
 check_All  

SELECT DISTINCT json_extract(METADATA, '$.duration') AS duration FROM EVENTS WHERE EVENT = 'JobExecution' AND JOB_ID = 8774052003 AND json_extract(METADATA, '$.duration') IS NOT NULL;
Thinking 
 Let's think step by step.
According to checked data structure, 'execution time' may be related to 'duration' in EVENT = 'JobExecution'.
According to '8774052003', JOB_ID may be used for filtering.
According to 'execution time', use json_extract(METADATA, '$.duration') to retrieve the value.

 

Tool Call 
 execute_sql SELECT DISTINCT json_extract(METADATA, '$.duration') AS duration FROM EVENTS WHERE EVENT = 'JobExecution' AND JOB_ID = 8774052003 AND js